In [76]:
import re
import torch
import pandas as pd

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

In [71]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("OpenMed/MedDialog")

train_df = dataset["train"].to_pandas()
train_df = train_df.drop_duplicates().reset_index(drop=True)

print(train_df.shape)

(218697, 3)


In [77]:
train_df["clean_text"] = train_df["patient_message"].str.lower()

train_df["clean_text"] = train_df["clean_text"].str.replace(r"\S+@\S+", " ", regex=True)
train_df["clean_text"] = train_df["clean_text"].str.replace(r"\b\d{10,}\b", " ", regex=True)
train_df["clean_text"] = train_df["clean_text"].str.replace(r"http\S+|www\S+", " ", regex=True)
train_df["clean_text"] = train_df["clean_text"].str.replace(r"[^a-z0-9\s]", " ", regex=True)
train_df["clean_text"] = train_df["clean_text"].str.replace(r"\s+", " ", regex=True).str.strip()

train_df[["patient_message","clean_text"]].head()

,patient_message,clean_text
126386,"What causes yellowness in eyes,numbness,loss o...",what causes yellowness in eyes numbness loss o...
49959,"What causes lethargy, chest pain, dizziness an...",what causes lethargy chest pain dizziness and ...
122760,Painful cut in the gum after using a new tooth...,painful cut in the gum after using a new tooth...
149329,"I have few small bumps over my face, how to ge...",i have few small bumps over my face how to get...
69044,Is it safe to take a flight after heart stroke...,is it safe to take a flight after heart stroke...


In [78]:
def annotate_intent(text):

    text = text.lower()

    emergency = [
        "emergency","urgent","severe","bleeding",
        "unconscious","stroke","heart attack",
        "difficulty breathing"
    ]

    prescription = [
        "tablet","medicine","drug","capsule",
        "dose","prescription","antibiotic"
    ]

    appointment = [
        "appointment","book","schedule","consult"
    ]

    lab = [
        "blood report","lab report","test report",
        "mri","ct scan","x ray","scan report"
    ]

    hospital = [
        "hospital address","location","timing",
        "contact number","open today"
    ]

    for i in emergency:
        if i in text:
            return "Emergency"

    for i in prescription:
        if i in text:
            return "Prescription"

    for i in appointment:
        if i in text:
            return "Appointment"

    for i in lab:
        if i in text:
            return "Lab_Report"

    for i in hospital:
        if i in text:
            return "Hospital_Info"

    return "Symptom"


train_df["intent"] = train_df["patient_message"].apply(annotate_intent)

In [79]:
label_encoder = LabelEncoder()

train_df["label"] = label_encoder.fit_transform(train_df["intent"])

print(label_encoder.classes_)

['Appointment' 'Emergency' 'Hospital_Info' 'Lab_Report' 'Prescription'
 'Symptom']


In [81]:
print(train_df.shape)

(10000, 6)


In [82]:
from sklearn.model_selection import train_test_split

# Create 10,000-sample subset only if needed
if len(train_df) > 10000:
    train_df, _ = train_test_split(
        train_df,
        train_size=10000,
        stratify=train_df["label"],
        random_state=42
    )
    print("10,000 sample subset created.")
else:
    print("Subset already exists.")

print(train_df.shape)

Subset already exists.
(10000, 6)


In [83]:
X = train_df["clean_text"]
y = train_df["label"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_train.shape)
print(X_val.shape)

(8000,)
(2000,)


In [84]:
tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")

train_encodings = tokenizer(
    X_train.tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

val_encodings = tokenizer(
    X_val.tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

In [85]:
class MedDialogDataset(Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        return {
            "input_ids": torch.tensor(self.encodings["input_ids"][idx]),
            "attention_mask": torch.tensor(self.encodings["attention_mask"][idx]),
            "labels": torch.tensor(self.labels.iloc[idx])
        }


train_dataset = MedDialogDataset(train_encodings, y_train)
val_dataset = MedDialogDataset(val_encodings, y_val)

In [86]:
BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [87]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSequenceClassification.from_pretrained(
    "emilyalsentzer/Bio_ClinicalBERT",
    num_labels=len(label_encoder.classes_)
)

model.to(device)

print(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the chec

cpu


In [88]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5
)

In [89]:
def train_one_epoch(model, loader, optimizer, device):

    model.train()

    total_loss = 0

    for batch in tqdm(loader):

        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [90]:
train_loss = train_one_epoch(
    model,
    train_loader,
    optimizer,
    device
)

print("Training Loss:", train_loss)

100%|██████████████████████████████████████████████████████████████████████████████| 500/500 [3:30:08<00:00, 25.22s/it]


Training Loss: 0.6142124010547996


In [91]:
#save model weights
torch.save(model.state_dict(),'hospital_triage_bert.pt')

#save label encoder classes
import joblib
joblib.dump(label_encoder,'label_encoder.pkl')

#save tokenizer
tokenizer.save_pretrained('tokenizer')

print('model saved successfully')

model saved successfully
